In [1]:
import numpy as np 
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from tqdm import tqdm

## CELL TYPE DIFFERENCE IN HM RNA

In [ ]:
scdata = sc.read_h5ad('/home/rsun@ZHANGroup.local/sly_data/Finished_analysis/metaneighbor_score/hmdata_full_harmony.h5ad')
scdata = scdata[scdata.obs['tissues'] == 'HM']
scdata

In [ ]:


metric_key = 'euclidean' #'cosine' #'cosine'

cell_res = {}
for cell in scdata.obs.celltype.unique():
    res_dic = {}
    adata = scdata[scdata.obs['celltype'] == cell,:] # celltype spcific data
    for species in ['CY','JT','T','QF','M']:
        species_idx = adata.obs['species'] == species 
        subdata = adata[species_idx,:] 

        # idx = np.arange(subdata.shape[0])
        # sel_idx = np.random.choice(idx, size = 500)
        meta_cell = subdata.X.toarray().mean(axis = 0)
        res_dic[species] = meta_cell 

    res_df = pd.DataFrame(res_dic).T
    sim_df = squareform(pdist(res_df, metric = metric_key))
    score = 2*(sim_df[:3,3:].mean()) - (sim_df[:3,:3].sum()/6 + sim_df[3:,3:].sum()/2)
    cell_res[cell] = score 

print(cell_res)

In [ ]:
diff = np.array(list(cell_res.values()))
tmp = np.exp(diff/4)
tmp = tmp/tmp.sum()
plt.bar(x =cell_res.keys(), height = tmp)
plt.xticks(rotation = 45)
plt.show()